## Case Study: Analyzing Food Delivery Performance

We have been given a core business question from our operations team:
> **"Which city had the highest average order value last month, among orders that were not cancelled?"**

To solve this, we must break this human English request down into logical data steps.

### The 3-Step Analytical Pipeline:
1. **Filter:** Kick out any orders that were cancelled (and focus on last month's timeframe).
2. **Aggregate:** Group the remaining orders by `city` and calculate the average (`AVG` / `mean`) order cost.
3. **Sort & Extract:** Sort the list from highest average cost to lowest, and take the top result.

In [69]:
import pandas as pd

# Load the Superstore-style dataset
data = {
    'order_id': ['O1','O2','O3','O4','O5','O6','O7','O8'],
    'city': ['Mumbai','Delhi','Mumbai','Bangalore','Delhi','Mumbai','Delhi','Bangalore'],
    'category': ['Tech','Furniture','Tech','Office','Tech','Furniture','Office','Tech'],
    'sales': [25000, 8000, 15000, 3000, 32000, 12000, 5500, 18000],
    'profit': [5000, -500, 3200, 800, 8000, 1200, 600, 4000],
    'status': ['completed','completed','completed','cancelled','completed','completed','cancelled','completed']
}
df = pd.DataFrame(data)
print(df)


  order_id       city   category  sales  profit     status
0       O1     Mumbai       Tech  25000    5000  completed
1       O2      Delhi  Furniture   8000    -500  completed
2       O3     Mumbai       Tech  15000    3200  completed
3       O4  Bangalore     Office   3000     800  cancelled
4       O5      Delhi       Tech  32000    8000  completed
5       O6     Mumbai  Furniture  12000    1200  completed
6       O7      Delhi     Office   5500     600  cancelled
7       O8  Bangalore       Tech  18000    4000  completed


In [70]:
df.groupby("status")["sales"].sum()

status
cancelled      8500
completed    110000
Name: sales, dtype: int64

In [71]:
sample=df[['order_id', 'status', 'profit']]
sample

,order_id,status,profit
0,O1,completed,5000
1,O2,completed,-500
2,O3,completed,3200
3,O4,cancelled,800
4,O5,completed,8000
5,O6,completed,1200
6,O7,cancelled,600
7,O8,completed,4000


In [72]:
df[df['sales'] > 10000][['order_id','sales']]

# select order_id, sales from data where sales > 10000

,order_id,sales
0,O1,25000
2,O3,15000
4,O5,32000
5,O6,12000
7,O8,18000


# Q1: Show all completed orders from Mumbai



In [73]:
# Pandas
result = df[(df['status'] == 'completed') & (df['city'] == 'Mumbai')]
print(result[['order_id','city','sales','profit']])

  order_id    city  sales  profit
0       O1  Mumbai  25000    5000
2       O3  Mumbai  15000    3200
5       O6  Mumbai  12000    1200


-- SQL equivalent

```
SELECT order_id, city, sales, profit
FROM orders
WHERE status = 'completed'
  AND city = 'Mumbai';


 # Q2. Top 3 orders by sales

In [74]:
# Pandas
top3 = df.sort_values('sales', ascending=False).head(3)
print(top3[['order_id','city','sales']])

  order_id       city  sales
4       O5      Delhi  32000
0       O1     Mumbai  25000
7       O8  Bangalore  18000


-- SQL equivalent

```
SELECT order_id, city, sales
FROM orders
ORDER BY sales DESC
LIMIT 3;


In [75]:
df[(df['sales'] > 10000) & (df['status'] == 'completed')]

,order_id,city,category,sales,profit,status
0,O1,Mumbai,Tech,25000,5000,completed
2,O3,Mumbai,Tech,15000,3200,completed
4,O5,Delhi,Tech,32000,8000,completed
5,O6,Mumbai,Furniture,12000,1200,completed
7,O8,Bangalore,Tech,18000,4000,completed


In [76]:
df.sort_values(['city', 'sales'], ascending=[True, False])
# All cities sorted A-Z; within each city, sales sorted highest first

,order_id,city,category,sales,profit,status
7,O8,Bangalore,Tech,18000,4000,completed
3,O4,Bangalore,Office,3000,800,cancelled
4,O5,Delhi,Tech,32000,8000,completed
1,O2,Delhi,Furniture,8000,-500,completed
6,O7,Delhi,Office,5500,600,cancelled
0,O1,Mumbai,Tech,25000,5000,completed
2,O3,Mumbai,Tech,15000,3200,completed
5,O6,Mumbai,Furniture,12000,1200,completed


**SQL Equvivalent**

SELECT *
FROM data
ORDER BY city ASC, sales DESC;

In [77]:
# --- isin filter ---
# Orders from metro cities only
metros = ['Mumbai', 'Delhi']
metro_orders = df[df['city'].isin(metros)]
print("\nMetro orders:")
print(metro_orders)


Metro orders:
  order_id    city   category  sales  profit     status
0       O1  Mumbai       Tech  25000    5000  completed
1       O2   Delhi  Furniture   8000    -500  completed
2       O3  Mumbai       Tech  15000    3200  completed
4       O5   Delhi       Tech  32000    8000  completed
5       O6  Mumbai  Furniture  12000    1200  completed
6       O7   Delhi     Office   5500     600  cancelled


SELECT *
FROM data
WHERE city IN ('Mumbai', 'Delhi');

In [78]:
~(df['category'] == "Furniture")

0     True
1    False
2     True
3     True
4     True
5    False
6     True
7     True
Name: category, dtype: bool

In [ ]:
# --- NOT filter ---
# All orders that are NOT Furniture
non_furniture_order = df[~(df['category'] == 'Furniture')]
print("\n non_furniture orders:", len(non_furniture_order))

In [79]:
non_furniture_order

,order_id,city,category,sales,profit,status
0,O1,Mumbai,Tech,25000,5000,completed
2,O3,Mumbai,Tech,15000,3200,completed
3,O4,Bangalore,Office,3000,800,cancelled
4,O5,Delhi,Tech,32000,8000,completed
6,O7,Delhi,Office,5500,600,cancelled
7,O8,Bangalore,Tech,18000,4000,completed


We have 8 orders.

Can you tell me:

* Total sales from Mumbai?
* Total sales from Delhi?
* Total sales from Bangalore?

**GroupBy**:

GroupBy means dividing data into groups based on a column and then performing calculations on each group.

In [80]:
df.groupby("city")

groupby() does NOT calculate anything.

It only separates rows into buckets.

Mumbai
----------------
O1   25000

O3   15000

O6   12000


Delhi
----------------
O2    8000

O5   32000

O7    5500


Bangalore
----------------
O4    3000

O8   18000

In [81]:
group_city = df.groupby("city")

print(type(group_city))

<class 'pandas.core.groupby.generic.DataFrameGroupBy'>


In [82]:
group_city.groups

{'Bangalore': [3, 7], 'Delhi': [1, 4, 6], 'Mumbai': [0, 2, 5]}

In [83]:
for city, group in group_city:
  print("City: ", city)
  print("Group:\n ", group)
  print(type(group))
  print("-------------------------------------------------")

City:  Bangalore
Group:
    order_id       city category  sales  profit     status
3       O4  Bangalore   Office   3000     800  cancelled
7       O8  Bangalore     Tech  18000    4000  completed
<class 'pandas.core.frame.DataFrame'>
-------------------------------------------------
City:  Delhi
Group:
    order_id   city   category  sales  profit     status
1       O2  Delhi  Furniture   8000    -500  completed
4       O5  Delhi       Tech  32000    8000  completed
6       O7  Delhi     Office   5500     600  cancelled
<class 'pandas.core.frame.DataFrame'>
-------------------------------------------------
City:  Mumbai
Group:
    order_id    city   category  sales  profit     status
0       O1  Mumbai       Tech  25000    5000  completed
2       O3  Mumbai       Tech  15000    3200  completed
5       O6  Mumbai  Furniture  12000    1200  completed
<class 'pandas.core.frame.DataFrame'>
-------------------------------------------------


In [84]:
"o1"+"o2"+"o3"

'o1o2o3'

### **Sum Sales By City**

In [85]:
dd=df.groupby("city")['sales']
dd.sum()

city
Bangalore    21000
Delhi        45500
Mumbai       52000
Name: sales, dtype: int64

In [86]:
df.groupby("city")["profit"].sum()

city
Bangalore    4800
Delhi        8100
Mumbai       9400
Name: profit, dtype: int64

### **Which city has most orders?**

In [87]:
dd=df.groupby("city")["order_id"].size()
dd.sort_values(ascending=False).head(1)

# size

city
Delhi    3
Name: order_id, dtype: int64

### **Which city generates highest average order value?**

In [88]:
dd=df.groupby("city")["sales"].mean()
dd.sort_values(ascending=False).head(1)


city
Mumbai    17333.333333
Name: sales, dtype: float64

### **Which category is most profitable?**

In [89]:
df.groupby("category")["profit"].sum()

category
Furniture      700
Office        1400
Tech         20200
Name: profit, dtype: int64

In [ ]:
city_sales = df.groupby("city")["sales"].sum()

for city, values in city_sales:
  print("City: \n", city)
  print("Values: \n", values)
  print("-----------------------------------------------------------------------")

# **Multiple Aggregations**

In [90]:
df.groupby('city')['sales'].agg(['sum', 'mean', 'count', 'max', 'min']).sort_values('sum', ascending=False)

,sum,mean,count,max,min
city,,,,,
Mumbai,52000,17333.333333,3,25000,12000
Delhi,45500,15166.666667,3,32000,5500
Bangalore,21000,10500.000000,2,18000,3000


In [91]:
df.groupby("city").agg(
    sales_sum=("sales","sum"),
    profit_mean=("profit","mean"),
    total_count=("order_id","count")
)

,sales_sum,profit_mean,total_count
city,,,
Bangalore,21000,2400.000000,2
Delhi,45500,2700.000000,3
Mumbai,52000,3133.333333,3


In [92]:
df.groupby("city").agg({
    "sales": "sum",
    "profit": "sum",
    "order_id": "count"
})

,sales,profit,order_id
city,,,
Bangalore,21000,4800,2
Delhi,45500,8100,3
Mumbai,52000,9400,3


### **Multiple GroupBy Columns**

### Sales by City and Category

Now management wants to know:

How much revenue does each category generate within each city?

In [97]:
cat_city = df.groupby(
    ["city", "category"]
)

KeyError: 'category'

In [ ]:
cat_city.groups


{('Bangalore', 'Office'): [3], ('Bangalore', 'Tech'): [7], ('Delhi', 'Furniture'): [1], ('Delhi', 'Office'): [6], ('Delhi', 'Tech'): [4], ('Mumbai', 'Furniture'): [5], ('Mumbai', 'Tech'): [0, 2]}

In [ ]:
df.groupby(
    ["city", "category"]
)["sales"].sum()

city       category 
Bangalore  Office        3000
           Tech         18000
Delhi      Furniture     8000
           Office        5500
           Tech         32000
Mumbai     Furniture    12000
           Tech         40000
Name: sales, dtype: int64

The answer is correct.

But it is difficult to compare categories across cities.

Let's improve the presentation.

In [98]:
import pandas as pd

df = pd.DataFrame({
    "city":["Chennai","Chennai","Mumbai","Mumbai","Delhi"],
    "sales":[100,200,300,400,500]
})

df

,city,sales
0,Chennai,100
1,Chennai,200
2,Mumbai,300
3,Mumbai,400
4,Delhi,500


In [99]:
df.groupby("city")["sales"].agg("mean")

city
Chennai    150.0
Delhi      500.0
Mumbai     350.0
Name: sales, dtype: float64

In [100]:
df.groupby("city")["sales"].transform("mean")

0    150.0
1    150.0
2    350.0
3    350.0
4    500.0
Name: sales, dtype: float64

In [101]:
df["city_avg"] = (
    df.groupby("city")["sales"]
      .transform("mean")
)

In [102]:
df

,city,sales,city_avg
0,Chennai,100,150.0
1,Chennai,200,150.0
2,Mumbai,300,350.0
3,Mumbai,400,350.0
4,Delhi,500,500.0


In [103]:
df

,city,sales,city_avg
0,Chennai,100,150.0
1,Chennai,200,150.0
2,Mumbai,300,350.0
3,Mumbai,400,350.0
4,Delhi,500,500.0


In [104]:
import pandas as pd

# Load
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
# Alternatively: df = pd.read_csv('titanic.csv')

print("Shape:", df.shape)
print("\n--- First look ---")
df.head()

Shape: (891, 12)

--- First look ---


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [108]:
print(df['Pclass'].value_counts())
df.groupby("Pclass")["Age"].agg("mean")

Pclass
3    491
1    216
2    184
Name: count, dtype: int64


Pclass
1    38.233441
2    29.877630
3    25.140620
Name: Age, dtype: float64

In [109]:
df.groupby("Pclass")["Age"].transform("mean")

0      25.140620
1      38.233441
2      25.140620
3      38.233441
4      25.140620
         ...    
886    29.877630
887    38.233441
888    25.140620
889    38.233441
890    25.140620
Name: Age, Length: 891, dtype: float64

In [110]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [111]:
df["Age"] = df["Age"].fillna(
    df.groupby("Pclass")["Age"]
           .transform("mean")
)

In [112]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
